# Two architectures that fail, and why each one fails

A dense model and a 1D convnet on the same problem. Both lose to the baseline, for two different and instructive reasons.

**Runs on:** CPU — about 10 minutes &nbsp;·&nbsp; **Slides:** [Chapter 13 — Timeseries Forecasting](../../../course-web-slides/ch13/index.html) &nbsp;·&nbsp; **Section:** 02 — Trying machine learning

---

## Setup

This notebook assumes the datasets from notebook 01. Re-run its cells, or import them if you have factored them out.

In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

sequence_length, n_features = 120, 14
NAIVE_MAE = 2.44

def train(model, name, epochs=10):
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    cb = [keras.callbacks.ModelCheckpoint(f"jena_{name}.keras",
                                          save_best_only=True)]
    h = model.fit(train_dataset, epochs=epochs,
                  validation_data=val_dataset, callbacks=cb, verbose=2)
    best = keras.models.load_model(f"jena_{name}.keras")
    mae = best.evaluate(test_dataset, verbose=0)[1]
    print(f"\n{name}: test MAE {mae:.2f} degC   (baseline {NAIVE_MAE})")
    return h, mae

## A dense model

In [ ]:
inputs = keras.Input(shape=(sequence_length, n_features))
x = layers.Flatten()(inputs)
x = layers.Dense(16, activation="relu")(x)
outputs = layers.Dense(1)(x)
dense_model = keras.Model(inputs, outputs)

h_dense, mae_dense = train(dense_model, "dense")

Expected output:

```
dense: test MAE 2.6x — 2.7x degC   (baseline 2.44)
```

**Worse than doing nothing.** The reason is instructive: `Flatten` destroys the time axis. The model receives 1,680 numbers with no indication that some of them are recent and some are five days old.

The simple solution is in its hypothesis space — it *could* learn to read the last temperature — but gradient descent has no reason to find that particular point among millions.

## A 1D convnet

In [ ]:
inputs = keras.Input(shape=(sequence_length, n_features))
x = layers.Conv1D(8, 24, activation="relu")(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 12, activation="relu")(x)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 6, activation="relu")(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1)(x)
conv_model = keras.Model(inputs, outputs)

h_conv, mae_conv = train(conv_model, "conv")

Expected output:

```
conv: test MAE 3.1x degC   (baseline 2.44)
```

**Worse still**, and for a different reason. Two properties that made convolution excellent on images are wrong here:

**Translation invariance.** A convolution treats a pattern the same wherever it occurs. But weather from five days ago is not equivalent to weather from an hour ago — and *recency is exactly what matters*.

**Pooling.** `MaxPooling1D` discards order within its window, and the most recent readings are the most informative ones being thrown away.

## The comparison

In [ ]:
plt.figure(figsize=(8, 4.6))
for h, name in [(h_dense, "dense"), (h_conv, "conv1d")]:
    plt.plot(h.history["val_mae"], lw=1.6, label=f"{name} — validation")
plt.axhline(NAIVE_MAE, color="k", ls="--", lw=1.4,
            label=f"naive baseline ({NAIVE_MAE})")
plt.xlabel("epoch"); plt.ylabel("validation MAE (degC)"); plt.legend()
plt.title("Neither architecture beats 'tomorrow is like today'")
plt.show()

## The general lesson

> A model's hypothesis space containing the right answer does not mean gradient descent will find it.

The dense model *could* have learned to output the last temperature. It did not, because nothing in its structure suggested that. **Architecture is a prior**, and the right prior for this problem is *the recent past matters more than the distant past* — which is precisely what a recurrent layer encodes, and what notebook 03 uses.

## One thing worth trying

In [ ]:
# Give the dense model the hint explicitly and watch what happens.
inputs = keras.Input(shape=(sequence_length, n_features))
recent = layers.Lambda(lambda t: t[:, -6:, :])(inputs)   # last six hours only
x = layers.Flatten()(recent)
x = layers.Dense(16, activation="relu")(x)
outputs = layers.Dense(1)(x)
recent_model = keras.Model(inputs, outputs)

h_recent, mae_recent = train(recent_model, "recent")

Cutting the input to the last six hours usually **improves** the dense model, sometimes to near the baseline. Removing information made it better — which tells you the problem was never capacity. It was the prior.

---

## What to take away

- `Flatten` destroys the time axis; the model cannot tell recent from distant.
- Convolution's translation invariance is **wrong** for forecasting, where recency is the signal.
- A hypothesis space containing the answer does not mean gradient descent will find it.
- Architecture is a prior. Choose one that matches the structure of the problem.